In [2]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [3]:
client.search_experiments()

[<Experiment: artifact_location='file:///c:/Users/vanthang/Documents/learn-ops/mlops-hands-on/02-experiment-tracking/mlruns/1', creation_time=1773405261479, experiment_id='1', last_update_time=1773405261479, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='file:///c:/Users/vanthang/Documents/learn-ops/mlops-hands-on/02-experiment-tracking/mlruns/0', creation_time=1773405261475, experiment_id='0', last_update_time=1773405261475, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
client.create_experiment(name="my-cool_experiment")

'2'

In [10]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [11]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 1154962ce72f44e7a0de0027f52ee44f, rmse: 6.3928
run id: 7d5294d3a40c4407b51c77f8a6e66d6f, rmse: 6.7423
run id: 8eb4476c0d3c4812ae3b91ecff58557b, rmse: 6.9053
run id: 2a9f8757bd954576936c4e93d79ef781, rmse: 6.9282


In [9]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [12]:
run_id="8eb4476c0d3c4812ae3b91ecff58557b"
model_uri=f"runs:/{run_id}/model"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

2026/03/13 21:25:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/13 21:25:46 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/03/13 21:25:46 WARNING mlflow.tracking._model_registry.fluent: Run with id 8eb4476c0d3c4812ae3b91ecff58557b has no artifacts at artifact path 'model', registering model based on models:/m-517d1ef094c54e699992bc76dea01793 instead
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1773411946826, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1773411946826, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='8eb4476c0d3c4812ae3b91ecff58557b', run_link=None, source='models:/m-517d1ef094c54e699992bc76dea01793', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [ ]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Staging
version: 2, stage: Production
version: 3, stage: None


C:\Users\vanthang\AppData\Local\Temp\ipykernel_14392\669935608.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [15]:
model_version = 3
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

C:\Users\vanthang\AppData\Local\Temp\ipykernel_14392\1957332551.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1773411946826, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1773412470857, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='8eb4476c0d3c4812ae3b91ecff58557b', run_link=None, source='models:/m-517d1ef094c54e699992bc76dea01793', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [16]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1773411946826, current_stage='Staging', deployment_job_state=None, description='The model version 3 was transitioned to Staging on 2026-03-13', last_updated_timestamp=1773412537684, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='8eb4476c0d3c4812ae3b91ecff58557b', run_link=None, source='models:/m-517d1ef094c54e699992bc76dea01793', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [19]:
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": np.sqrt(mean_squared_error(y_test, y_pred))}

In [20]:
df = read_dataframe("data/green_tripdata_2021-03.parquet")

In [21]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'c:\\Users\\vanthang\\Documents\\learn-ops\\mlops-hands-on\\02-experiment-tracking\\preprocessor'

In [22]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [23]:
X_test = preprocess(df, dv)

In [24]:
target = "duration"
y_test = df[target].values

In [25]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: total: 1.61 s
Wall time: 1.86 s


{'rmse': np.float64(6.659623830022513)}

In [27]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: total: 25.5 s
Wall time: 2.41 s


{'rmse': np.float64(6.315126327491971)}

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)